# Data Analysis and Visualisation

## Lab 2: Group and create

### Preliminaries

In [ ]:
import numpy as np
import pandas as pd

listings = pd.read_csv("data/listings.csv")

### Part 1

#### Task 1: Count property types

In [ ]:
print(listings['property_type'].value_counts().head(5))
share = listings['property_type'].value_counts(normalize=True)['Entire rental unit']
print(round(share, 3))

property_type
Entire rental unit             2442
Entire home                     569
Private room in rental unit     491
Entire condo                    339
Entire villa                    149
Name: count, dtype: int64
0.534


#### Task 2: The smallest neighbourhoods

In [ ]:
# The list is sorted in descending order, so the smallest groups are at the end
print(listings['neighbourhood_cleansed'].value_counts().tail(3))

neighbourhood_cleansed
Rinkeby-Tensta    108
Skärholmens        96
Spånga-Tensta      89
Name: count, dtype: int64


#### Task 3: Average beds per room type

In [ ]:
print(listings.groupby('room_type')['beds'].mean().round(2))
display(listings.groupby('room_type')['beds'].agg(['count', 'mean']).round(2))

room_type
Entire home/apt    2.48
Hotel room         1.00
Private room       1.48
Shared room        4.58
Name: beds, dtype: float64


,count,mean
room_type,,
Entire home/apt,3680,2.48
Hotel room,3,1.00
Private room,766,1.48
Shared room,24,4.58


I trust the "Hotel room" average the least. It is based on only 3 listings with a known number of beds, so a single listing would change it a lot. The "Entire home/apt" average rests on 3680 listings, which is much safer.

#### Task 4: Reviews per neighbourhood

In [ ]:
reviews = (listings.groupby('neighbourhood_cleansed')['number_of_reviews']
           .agg(['count', 'mean', 'median'])
           .round(2)
           .sort_values('median'))
display(reviews)

,count,mean,median
neighbourhood_cleansed,,,
Skärholmens,96,21.29,3.0
Farsta,177,18.18,4.0
Hässelby-Vällingby,148,14.94,4.0
Rinkeby-Tensta,108,16.44,5.0
Bromma,281,19.16,5.0
Enskede-Årsta-Vantörs,445,22.55,7.0
Spånga-Tensta,89,17.01,7.0
Hägersten-Liljeholmens,464,22.92,8.0
Skarpnäcks,221,22.55,8.0


Lowest median: Skärholmens (3). Highest: Södermalms (17).

In Södermalms the mean (59.82) is much bigger than the median. A few listings with a very large number of reviews pull the mean up, while the median only looks at the middle listing. So the median is the better "typical" value here.

#### Task 5: Summary table

In [ ]:
summary = (listings.groupby('room_type')
           .agg(n_listings=('id', 'count'), 
                avg_beds=('beds', 'mean'), 
                max_reviews=('number_of_reviews', 'max'))
           .round(2))
display(summary)

,n_listings,avg_beds,max_reviews
room_type,,,
Entire home/apt,3760,2.48,1293
Hotel room,3,1.00,79
Private room,787,1.48,867
Shared room,24,4.58,510


#### Task 6: Two categories at once

In [ ]:
ct = (pd.crosstab(listings['neighbourhood_cleansed'],
                  listings['host_is_superhost'],
                  normalize='index') * 100).round(1)
print(ct['t'].sort_values())

neighbourhood_cleansed
Bromma                     7.1
Farsta                     9.0
Kungsholmens               9.8
Enskede-Årsta-Vantörs     11.7
Hässelby-Vällingby        12.2
Hägersten-Liljeholmens    12.5
Rinkeby-Tensta            13.0
Skarpnäcks                13.1
Skärholmens               15.6
Östermalms                16.8
Norrmalms                 19.0
Älvsjö                    19.6
Spånga-Tensta             22.5
Södermalms                23.8
Name: t, dtype: float64


The highest share of superhosts is in Södermalm with 23.8%, and the lowest in Bromma with 7.1%.

#### Task 7: Create new columns

In [ ]:
listings['has_reviews'] = listings['number_of_reviews'] > 0
print("Number of listings without reviews:", (~listings['has_reviews']).sum())

listings['popularity'] = np.where(listings['number_of_reviews'] >= 50, 'popular', 'other')
display(listings['popularity'].value_counts())

Number of listings without reviews: 644


popularity
other      3805
popular     769
Name: count, dtype: int64

#### Task 8: Turn numbers into groups

In [ ]:
first_try = pd.cut(listings['minimum_nights'], bins=[1, 6, 29, 500])
print(first_try.isna().sum())

1331


With `bins=[1, 6, 29, 500]` the groups are `(1, 6]`, `(6, 29]`, `(29, 500]`: the left edge is left out, so a listing with exactly 1 night as minimum stay is therefore in no group. That gives 1327 missing, plus 4 listings that have no minimum_nights at all = 1331. Adding `0` as the first edge gives `(0, 1]` so that 1 night gets its own group.

In [ ]:
listings['stay'] = pd.cut(listings['minimum_nights'],
                          bins=[0, 1, 6, 29, 500],
                          labels=['1 night', '2-6 nights', '7-29 nights', '30+ nights'])
display(listings['stay'].value_counts().sort_index())
print("Listings with no minimum stay:", listings['stay'].isna().sum())

stay
1 night        1327
2-6 nights     2661
7-29 nights     430
30+ nights      152
Name: count, dtype: int64

Listings with no minimum stay: 4
